# 05 · Definición provisional de vulnerabilidad por patrimonio (Q1)

**Objetivo:** prototipar el cuartil inferior por ronda nominal sin confundir patrimonio con ingreso. El outcome definitivo para predicción deberá ser el **estado futuro**, no el estado actual.

**AVISO:** aquí los cuartiles se calculan por ola sobre la muestra filtrada (diagnóstico, no target definitivo). Antes de pronóstico definir la población de referencia, uso correcto de pesos y tratamiento de cortes fuera de muestra.


In [ ]:
from tesis_enasem.config import RAW_DIR, ROUND_TO_YEAR
from tesis_enasem.io import load_stata
from tesis_enasem.panel import make_person_id, add_year, filter_population, followup_pairs
from tesis_enasem.outcomes import wealth_q1_cutoff, label_by_wave
from tesis_enasem.transforms import add_asinh
from tesis_enasem.missing import numeric_enasem
import pandas as pd

raw=load_stata(RAW_DIR/'simpleMHAS.dta',preserve_missing=True)
raw['id']=make_person_id(raw)
df=filter_population(add_year(raw, ROUND_TO_YEAR))
df=add_asinh(df, 'imp_neto')


## Cuartiles por año nominal
Los factores individuales son por ola; este ejemplo usa cuantiles no ponderados hasta resolver el diseño de ponderación.


In [ ]:
tabla=[]
for ano,sub in df.groupby('ano'):
    x=numeric_enasem(sub['imp_neto']).dropna()
    tabla.append({'ano':int(ano),'N_patrimonio':len(x),'Q1':x.quantile(.25),'mediana':x.median(),'Q3':x.quantile(.75)})
display(pd.DataFrame(tabla).round(2))


## Vulnerabilidad y pares temporales
No sustituir las etiquetas futuras ausentes por 0. `followup_pairs` exige olas consecutivas, pero **no** años consecutivos.


In [ ]:
labeled=label_by_wave(df)
paired=followup_pairs(labeled,'vulnerable_q1')
resumen=paired.groupby('ano')['outcome_futuro'].count().to_frame('observaciones_futuras')
display(resumen)


**Pendiente CP3:** comprobar definiciones alternativas, cortes Q1 reproducibles sin fuga del test, deflactor, identificación de evento incidente entre personas previamente no vulnerables, transición/persistencia.
